# Variant comparison: perishable goods
Test two variants, one without perishable goods ($\beta=0$ and no $T_{max}$)

# Model m0 (Full)

In [1]:
# Imports
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [2]:
# Load data
DATA_REAL = "../data/real/"
DATA_SYN = "../data/synthetic/"

distance_df = pd.read_csv(DATA_REAL + "distance_matrix.csv", index_col=0)
time_df = pd.read_csv(DATA_REAL + "time_matrix.csv", index_col=0)

cafes_df = pd.read_csv(DATA_SYN + "cafes.csv")
demands_df = pd.read_csv(DATA_SYN + "demands.csv")
products_df = pd.read_csv(DATA_SYN + "products.csv")
vans_df = pd.read_csv(DATA_SYN + "vans.csv")
depot_df = pd.read_csv(DATA_SYN + "depot.csv")
perish_df = pd.read_csv(DATA_SYN + "perishability_params.csv")

In [3]:
# Sets
depot_name = depot_df.loc[0, "name"]

cafes = cafes_df["cafe_name"].tolist()
nodes = [depot_name] + cafes

K = vans_df["van_id"].tolist()
P = products_df["product_id"].tolist()

milk_products = products_df.loc[
    products_df["category"] == "milk", "product_id"
].tolist()

A = [(i, j) for i in nodes for j in nodes if i != j]

In [4]:
# Parameters
# Demand d[i,p]
demand_raw = {
    (row["cafe_id"], row["product_id"]): row["daily_demand"]
    for _, row in demands_df.iterrows()
}

cafe_id_to_name = dict(zip(cafes_df["cafe_id"], cafes_df["cafe_name"]))

d = {}
for cafe_id, cafe_name in cafe_id_to_name.items():
    for p in P:
        d[cafe_name, p] = demand_raw.get((cafe_id, p), 0)

# Product parameters
w = dict(zip(products_df["product_id"], products_df["weight_per_unit_kg"]))
revenue = dict(zip(products_df["product_id"], products_df["revenue_per_unit"]))
cost = dict(zip(products_df["product_id"], products_df["cost_per_unit"]))
margin = {
    p: revenue[p] - cost[p]
    for p in P
}

# Van parameters
Q = dict(zip(vans_df["van_id"], vans_df["capacity_kg"]))
F = dict(zip(vans_df["van_id"], vans_df["fuel_cost_per_km"]))

# Distance and time
D = {}
T = {}

for i, j in A:
    D[i, j] = float(distance_df.loc[i, j])
    T[i, j] = float(time_df.loc[i, j]) / 60.0  # minutes to hours

# Perishability parameters
perish_params = dict(zip(perish_df["parameter"], perish_df["value"]))

T_max = float(perish_params["T_max_hours"])
loading_time = float(perish_params["loading_time_minutes"]) / 60.0
service_time = float(perish_params["service_time_minutes"]) / 60.0

# Long route penalty coefficient
beta = 5.0

# Big-M values
M_milk = sum(d[i, p] for i in cafes for p in milk_products)
M_time = 1000

In [5]:
perish_params 

{'T_max_hours': 3.0,
 'T_max_hours_alt1': 2.0,
 'T_max_hours_alt2': 4.0,
 'milk_spoilage_cost_per_hour': 0.15,
 'loading_time_minutes': 15.0,
 'service_time_minutes': 5.0}

In [6]:
# Build model
m0 = gp.Model("profit_max_vrp")

# Routing variable
x = m0.addVars(A, K, vtype=GRB.BINARY, name="x")

# Service variables
y = m0.addVars(cafes, K, vtype=GRB.BINARY, name="y")
z = m0.addVars(cafes, vtype=GRB.BINARY, name="z")

# Delivery quantity
q = m0.addVars(cafes, P, K, lb=0, vtype=GRB.CONTINUOUS, name="q")

# Route duration
R = m0.addVars(K, lb=0, vtype=GRB.CONTINUOUS, name="R")

# Milk indicator
g = m0.addVars(K, vtype=GRB.BINARY, name="g")

# MTZ variables
u = m0.addVars(cafes, K, lb=0, vtype=GRB.CONTINUOUS, name="u")


Set parameter Username
Set parameter LicenseID to value 2824592
Academic license - for non-commercial use only - expires 2027-05-20


In [7]:
# Objective
product_margin = gp.quicksum(
    margin[p] * q[i, p, k]
    for i in cafes
    for p in P
    for k in K
)

transport_cost = gp.quicksum(
    F[k] * D[i, j] * x[i, j, k]
    for (i, j) in A
    for k in K
)

route_penalty = gp.quicksum(
    beta * R[k]
    for k in K
)

m0.setObjective(
    product_margin - transport_cost - route_penalty,
    GRB.MAXIMIZE
)


In [8]:
# 1. Each cafe served by at most one van
m0.addConstrs(
    (gp.quicksum(y[i, k] for k in K) == z[i]
     for i in cafes),
    name="serve_once"
)

# 2. Demand satisfaction
m0.addConstrs(
    (gp.quicksum(q[i, p, k] for k in K) == d[i, p] * z[i]
     for i in cafes
     for p in P),
    name="demand_satisfaction"
)

# 3. Deliveries only if van serves cafe
m0.addConstrs(
    (q[i, p, k] <= d[i, p] * y[i, k]
     for i in cafes
     for p in P
     for k in K),
    name="delivery_if_served"
)

# 4. Vehicle capacity
m0.addConstrs(
    (gp.quicksum(w[p] * q[i, p, k]
                 for i in cafes
                 for p in P) <= Q[k]
     for k in K),
    name="capacity"
)

# 5. Flow conservation
m0.addConstrs(
    (gp.quicksum(x[i, j, k] for j in nodes if j != i) == y[i, k]
     for i in cafes
     for k in K),
    name="flow_out"
)

m0.addConstrs(
    (gp.quicksum(x[j, i, k] for j in nodes if j != i) == y[i, k]
     for i in cafes
     for k in K),
    name="flow_in"
)

# 6. Depot departure and return
m0.addConstrs(
    (gp.quicksum(x[depot_name, j, k] for j in cafes) <= 1
     for k in K),
    name="depot_departure"
)

m0.addConstrs(
    (gp.quicksum(x[i, depot_name, k] for i in cafes) <= 1
     for k in K),
    name="depot_return"
)

m0.addConstrs(
    (gp.quicksum(x[depot_name, j, k] for j in cafes)
     ==
     gp.quicksum(x[i, depot_name, k] for i in cafes)
     for k in K),
    name="depot_balance"
)

# 7. MTZ subtour elimination
n_cafes = len(cafes)

m0.addConstrs(
    (u[i, k] - u[j, k] + n_cafes * x[i, j, k]
     <= n_cafes - 1
     for i in cafes
     for j in cafes
     if i != j
     for k in K),
    name="mtz"
)

m0.addConstrs(
    (u[i, k] <= n_cafes * y[i, k]
     for i in cafes
     for k in K),
    name="mtz_upper"
)

m0.addConstrs(
    (u[i, k] >= y[i, k]
     for i in cafes
     for k in K),
    name="mtz_lower"
)

# 8. Route duration
m0.addConstrs(
    (R[k] ==
     loading_time * gp.quicksum(x[depot_name, j, k] for j in cafes)
     +
     gp.quicksum(T[i, j] * x[i, j, k] for (i, j) in A)
     +
     service_time * gp.quicksum(y[i, k] for i in cafes)
     for k in K),
    name="route_duration"
)

# 9. Milk indicator
m0.addConstrs(
    (gp.quicksum(q[i, p, k]
                 for i in cafes
                 for p in milk_products)
     <= M_milk * g[k]
     for k in K),
    name="milk_indicator"
)

# 10. Maximum route duration if carrying milk
m0.addConstrs(
    (R[k] <= T_max + M_time * (1 - g[k])
     for k in K),
    name="milk_time_limit"
)

{'van_1': <gurobi.Constr *Awaiting Model Update*>,
 'van_2': <gurobi.Constr *Awaiting Model Update*>,
 'van_3': <gurobi.Constr *Awaiting Model Update*>,
 'van_4': <gurobi.Constr *Awaiting Model Update*>,
 'van_5': <gurobi.Constr *Awaiting Model Update*>}

In [9]:
m0.Params.OutputFlag = 1
m0.Params.TimeLimit = 300

m0.optimize()


Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 24.6.0 24G624)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 7787 rows, 7524 columns and 41980 nonzeros (Max)
Model fingerprint: 0x0f2725ff
Model has 7145 linear objective coefficients
Variable types: 1365 continuous, 6159 integer (6159 binary)
Coefficient statistics:
  Matrix range     [7e-03, 1e+03]
  Objective range  [7e-02, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+03]

Found heuristic solution: objective -0.0000000
Presolve removed 1433 rows and 1229 columns
Presolve time: 0.19s
Presolved: 6354 rows, 6295 columns, 36505 nonzeros
Variable types: 170 continuous, 6125 integer (6125 binary)

Root relaxation: objective 2.888624e+03, 619 iterations, 0.02 seconds (0.03 work units)

    Nodes    |    Current

In [10]:
if m0.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:

    print("\nObjective value:", m0.ObjVal)

    print("\nServed cafes:")
    for i in cafes:
        if z[i].X > 0.5:
            print(f"- {i}")

    print("\nVan routes:")

    for k in K:
        used = sum(x[depot_name, j, k].X for j in cafes)

        if used > 0.5:
            print(f"\n{k}")
            print(f"Route duration: {R[k].X:.2f} hours")
            print(f"Carries milk: {round(g[k].X)}")

            route = [depot_name]
            current = depot_name

            while True:
                next_nodes = [
                    j for j in nodes
                    if j != current and x[current, j, k].X > 0.5
                ]

                if not next_nodes:
                    break

                next_node = next_nodes[0]
                route.append(next_node)

                if next_node == depot_name:
                    break

                current = next_node

            print(" -> ".join(route))

            total_weight = sum(
                w[p] * q[i, p, k].X
                for i in cafes
                for p in P
            )

            print(f"Load: {total_weight:.2f} kg / {Q[k]} kg")

            print("Deliveries:")
            for i in cafes:
                delivered_any = False
                lines = []

                for p in P:
                    val = q[i, p, k].X
                    if val > 1e-6:
                        delivered_any = True
                        lines.append(f"  {p}: {val:.2f}")

                if delivered_any:
                    print(f" {i}")
                    for line in lines:
                        print(line)

else:
    print("Model was not solved successfully.")
    print("Status:", m0.status)


Objective value: 2875.1354533333315

Served cafes:
- Proud Mary Coffee
- Patricia Coffee Brewers
- Seven Seeds Coffee Roasters
- Industry Beans Fitzroy
- Market Lane Coffee HQ
- ST. ALi Coffee Roasters
- Code Black Coffee Brunswick
- Axil Coffee Roasters
- Dukes Coffee Roasters
- Rumble Coffee Roasters
- Wide Open Road
- Aunty Peg's
- Brother Baba Budan
- Lune Croissanterie Fitzroy
- Small Batch Roasting Co.
- Padre Coffee Brunswick East
- Jasper Coffee Fitzroy
- Everyday Coffee
- MAKER Coffee South Yarra
- Square Lane Coffee
- Disciple Roasters
- Coffee Supreme
- Market Lane Prahran Market
- Veneziano Coffee Richmond
- Five Senses Coffee
- Standing Room Fitzroy North
- Bench Coffee Co. Roastery
- Path Melbourne
- Little Rogue
- Vacation Coffee CBD
- Market Lane Carlton
- Traveller Coffee
- Core Roasters
- Project Zero

Van routes:

van_2
Route duration: 1.89 hours
Carries milk: 1
Peter Hall Building UniMelb -> Lune Croissanterie Fitzroy -> Industry Beans Fitzroy -> Standing Room Fitz

In [11]:
# Capture m0 variables before m1 overwrites the names x, y, z, q, R, g
m0_vars = dict(
    cafes=cafes, nodes=nodes, K=K, P=P, A=A, depot_name=depot_name,
    x=x, y=y, z=z, q=q, R=R, g=g,
    w=w, margin=margin, F=F, D=D, beta=5.0
)


# Model m1 (No perishability)

In [12]:
# Build model
m1 = gp.Model("profit_max_vrp_no_perish")

# Routing variable
x = m1.addVars(A, K, vtype=GRB.BINARY, name="x")

# Service variables
y = m1.addVars(cafes, K, vtype=GRB.BINARY, name="y")
z = m1.addVars(cafes, vtype=GRB.BINARY, name="z")

# Delivery quantity
q = m1.addVars(cafes, P, K, lb=0, vtype=GRB.CONTINUOUS, name="q")

# Route duration
R = m1.addVars(K, lb=0, vtype=GRB.CONTINUOUS, name="R")

# Milk indicator
g = m1.addVars(K, vtype=GRB.BINARY, name="g")

# MTZ variables
u = m1.addVars(cafes, K, lb=0, vtype=GRB.CONTINUOUS, name="u")

In [13]:
# Long route penalty coefficient
beta = 0
# Perishability
perish_params_none = perish_params
perish_params_none['T_max_hours'] = 0
perish_params_none['T_max_hours_alt1'] = 0
perish_params_none['T_max_hours_alt2'] = 0
perish_params_none

product_margin = gp.quicksum(
    margin[p] * q[i, p, k]
    for i in cafes
    for p in P
    for k in K
)

transport_cost = gp.quicksum(
    F[k] * D[i, j] * x[i, j, k]
    for (i, j) in A
    for k in K
)

route_penalty = gp.quicksum(
    beta * R[k]
    for k in K
)


In [14]:
# Objective
m1.setObjective(
    product_margin - transport_cost - route_penalty,
    GRB.MAXIMIZE
)

In [15]:
# 1. Each cafe served by at most one van
m1.addConstrs(
    (gp.quicksum(y[i, k] for k in K) == z[i]
     for i in cafes),
    name="serve_once"
)

# 2. Demand satisfaction
m1.addConstrs(
    (gp.quicksum(q[i, p, k] for k in K) == d[i, p] * z[i]
     for i in cafes
     for p in P),
    name="demand_satisfaction"
)

# 3. Deliveries only if van serves cafe
m1.addConstrs(
    (q[i, p, k] <= d[i, p] * y[i, k]
     for i in cafes
     for p in P
     for k in K),
    name="delivery_if_served"
)

# 4. Vehicle capacity
m1.addConstrs(
    (gp.quicksum(w[p] * q[i, p, k]
                 for i in cafes
                 for p in P) <= Q[k]
     for k in K),
    name="capacity"
)

# 5. Flow conservation
m1.addConstrs(
    (gp.quicksum(x[i, j, k] for j in nodes if j != i) == y[i, k]
     for i in cafes
     for k in K),
    name="flow_out"
)

m1.addConstrs(
    (gp.quicksum(x[j, i, k] for j in nodes if j != i) == y[i, k]
     for i in cafes
     for k in K),
    name="flow_in"
)

# 6. Depot departure and return
m1.addConstrs(
    (gp.quicksum(x[depot_name, j, k] for j in cafes) <= 1
     for k in K),
    name="depot_departure"
)

m1.addConstrs(
    (gp.quicksum(x[i, depot_name, k] for i in cafes) <= 1
     for k in K),
    name="depot_return"
)

m1.addConstrs(
    (gp.quicksum(x[depot_name, j, k] for j in cafes)
     ==
     gp.quicksum(x[i, depot_name, k] for i in cafes)
     for k in K),
    name="depot_balance"
)

# 7. MTZ subtour elimination
n_cafes = len(cafes)

m1.addConstrs(
    (u[i, k] - u[j, k] + n_cafes * x[i, j, k]
     <= n_cafes - 1
     for i in cafes
     for j in cafes
     if i != j
     for k in K),
    name="mtz"
)

m1.addConstrs(
    (u[i, k] <= n_cafes * y[i, k]
     for i in cafes
     for k in K),
    name="mtz_upper"
)

m1.addConstrs(
    (u[i, k] >= y[i, k]
     for i in cafes
     for k in K),
    name="mtz_lower"
)

# 8. Route duration
m1.addConstrs(
    (R[k] ==
     loading_time * gp.quicksum(x[depot_name, j, k] for j in cafes)
     +
     gp.quicksum(T[i, j] * x[i, j, k] for (i, j) in A)
     +
     service_time * gp.quicksum(y[i, k] for i in cafes)
     for k in K),
    name="route_duration"
)

# 9. Milk indicator
m1.addConstrs(
    (gp.quicksum(q[i, p, k]
                 for i in cafes
                 for p in milk_products)
     <= M_milk * g[k]
     for k in K),
    name="milk_indicator"
)

# 10. Maximum route duration if carrying milk
m1.addConstrs(
    (R[k] <= T_max + M_time * (1 - g[k])
     for k in K),
    name="milk_time_limit"
)

{'van_1': <gurobi.Constr *Awaiting Model Update*>,
 'van_2': <gurobi.Constr *Awaiting Model Update*>,
 'van_3': <gurobi.Constr *Awaiting Model Update*>,
 'van_4': <gurobi.Constr *Awaiting Model Update*>,
 'van_5': <gurobi.Constr *Awaiting Model Update*>}

In [16]:
m1.Params.OutputFlag = 1
m1.Params.TimeLimit = 300

m1.optimize()


Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 24.6.0 24G624)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  300

Optimize a model with 7787 rows, 7524 columns and 41980 nonzeros (Max)
Model fingerprint: 0xf5f99460
Model has 7140 linear objective coefficients
Variable types: 1365 continuous, 6159 integer (6159 binary)
Coefficient statistics:
  Matrix range     [7e-03, 1e+03]
  Objective range  [7e-02, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+03]

Found heuristic solution: objective -0.0000000
Presolve removed 1433 rows and 1229 columns
Presolve time: 0.21s
Presolved: 6354 rows, 6295 columns, 36505 nonzeros
Variable types: 170 continuous, 6125 integer (6125 binary)

Root relaxation: objective 2.909678e+03, 711 iterations, 0.02 seconds (0.03 work units)

    Nodes    |    Current

In [17]:
if m1.status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:

    print("\nObjective value:", m1.ObjVal)

    print("\nServed cafes:")
    for i in cafes:
        if z[i].X > 0.5:
            print(f"- {i}")

    print("\nVan routes:")

    for k in K:
        used = sum(x[depot_name, j, k].X for j in cafes)

        if used > 0.5:
            print(f"\n{k}")
            print(f"Route duration: {R[k].X:.2f} hours")
            print(f"Carries milk: {round(g[k].X)}")

            route = [depot_name]
            current = depot_name

            while True:
                next_nodes = [
                    j for j in nodes
                    if j != current and x[current, j, k].X > 0.5
                ]

                if not next_nodes:
                    break

                next_node = next_nodes[0]
                route.append(next_node)

                if next_node == depot_name:
                    break

                current = next_node

            print(" -> ".join(route))

            total_weight = sum(
                w[p] * q[i, p, k].X
                for i in cafes
                for p in P
            )

            print(f"Load: {total_weight:.2f} kg / {Q[k]} kg")

            print("Deliveries:")
            for i in cafes:
                delivered_any = False
                lines = []

                for p in P:
                    val = q[i, p, k].X
                    if val > 1e-6:
                        delivered_any = True
                        lines.append(f"  {p}: {val:.2f}")

                if delivered_any:
                    print(f" {i}")
                    for line in lines:
                        print(line)

else:
    print("Model was not solved successfully.")
    print("Status:", m1.status)


Objective value: 2903.1338649999984

Served cafes:
- Proud Mary Coffee
- Patricia Coffee Brewers
- Seven Seeds Coffee Roasters
- Industry Beans Fitzroy
- Market Lane Coffee HQ
- ST. ALi Coffee Roasters
- Code Black Coffee Brunswick
- Axil Coffee Roasters
- Dukes Coffee Roasters
- Rumble Coffee Roasters
- Wide Open Road
- Aunty Peg's
- Brother Baba Budan
- Lune Croissanterie Fitzroy
- Small Batch Roasting Co.
- Padre Coffee Brunswick East
- Jasper Coffee Fitzroy
- Everyday Coffee
- MAKER Coffee South Yarra
- Square Lane Coffee
- Disciple Roasters
- Coffee Supreme
- Market Lane Prahran Market
- Veneziano Coffee Richmond
- Five Senses Coffee
- Standing Room Fitzroy North
- Bench Coffee Co. Roastery
- Path Melbourne
- Little Rogue
- Vacation Coffee CBD
- Market Lane Carlton
- Traveller Coffee
- Core Roasters
- Project Zero

Van routes:

van_1
Route duration: 1.56 hours
Carries milk: 1
Peter Hall Building UniMelb -> Lune Croissanterie Fitzroy -> Industry Beans Fitzroy -> Standing Room Fitz

In [18]:
# Capture m1 variables (beta=0, no perishability penalty)
m1_vars = dict(
    cafes=cafes, nodes=nodes, K=K, P=P, A=A, depot_name=depot_name,
    x=x, y=y, z=z, q=q, R=R, g=g,
    w=w, margin=margin, F=F, D=D, beta=0
)


### Comparison



In [19]:
def extract_results(model, label, cafes, nodes, K, P, A, depot_name,
                     x, y, z, q, R, g, w, margin, F, D, beta):
    """Extract key metrics from a solved Gurobi model into a results dict."""
    result = {'Model': label}
    if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT] and model.SolCount > 0:
        tot_margin    = sum(margin[p] * q[i, p, k].X for i in cafes for p in P for k in K)
        tot_transport = sum(F[k] * D[i, j] * x[i, j, k].X for (i, j) in A for k in K)
        tot_dist      = sum(D[i, j] * x[i, j, k].X for (i, j) in A for k in K)
        tot_penalty   = sum(beta * R[k].X for k in K)
        result['Profit ($)']     = round(tot_margin - tot_transport - tot_penalty, 2)
        result['Margin ($)']     = round(tot_margin, 2)
        result['Transport ($)']  = round(tot_transport, 2)
        result['Penalty ($)']    = round(tot_penalty, 2)
        result['Distance (km)']  = round(tot_dist, 2)
        result['Cafes Served']   = sum(1 for i in cafes if z[i].X > 0.5)
        result['Solve Time (s)'] = round(model.Runtime, 2)
        result['MIP Gap (%)']    = round(model.MIPGap * 100, 2)
        routes = []
        for k in K:
            if sum(x[depot_name, j, k].X for j in cafes) > 0.5:
                route = [depot_name]
                current = depot_name
                visited = set()
                while True:
                    nxts = [j for j in nodes if j != current
                            and j not in visited and x[current, j, k].X > 0.5]
                    if not nxts:
                        break
                    nxt = nxts[0]
                    route.append(nxt)
                    if nxt == depot_name:
                        break
                    visited.add(nxt)
                    current = nxt
                routes.append(f'{k}: {" -> ".join(route)} ({R[k].X:.2f}h)')
        result['Routes'] = ' | '.join(routes)
    else:
        result['Profit ($)'] = None
        result['Routes']     = ''
    return result


In [ ]:
# Collect results for each variant.
# m0_vars / m1_vars are captured in the cells immediately after each model's solve.
r0 = extract_results(m0, 'm0 (With perishability)',  **m0_vars)
r1 = extract_results(m1, 'm1 (No perishability)', **m1_vars)

comparison = pd.DataFrame([r0, r1])
print(comparison.to_string(index=False))

comparison.to_csv('../results/perishability_results.csv', index=False)
print('\nResults saved to perishability_results.csv')


                  Model  Profit ($)  Margin ($)  Transport ($)  Penalty ($)  Distance (km)  Cafes Served  Solve Time (s)  MIP Gap (%)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   